# Test Contrastive Model Embedding Visualization

This notebook tests the UMAP embedding visualization functionality for the ContrastiveEngine.

You can:
- Load a trained model from checkpoint
- Use a randomly initialized model
- Test the visualization with different numbers of users

In [ ]:
import sys
sys.path.append('../')

import torch
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from torch.utils.data import DataLoader

from edamame.model.efficientnet_contrastive import EfficientNetContrastive
from edamame.model.loss.contrastive import NCELoss
from edamame.engine.contrastive import ContrastiveEngine
from edamame.data.dataset import EDAtaNoLabel

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

import matplotlib
matplotlib.use('qtagg') 

## Configuration

In [4]:
# Choose mode: 'checkpoint' or 'random'
MODE = 'random'  # Change to 'checkpoint' to load a trained model

# Checkpoint path (only used if MODE='checkpoint')
CHECKPOINT_PATH = '../checkpoints.nosync/[EfficientNet Contrastive, User-pair] Explore: lr, batch size and dimensionality/epoch=53-val_loss=6.1675'

# Data configuration
DATA_BASE_PATH = '../data.nosync/60s'  # Path to data folder
DATASET_NAME = 'BiHeartS'  # Dataset to load: 'BiHeartS', 'APSYNC', 'CLUES', etc.
MAX_SAMPLES = 20  # Limit number of samples for faster testing

# Model configuration
SEQ_LEN = 240  # Sequence length for 30s data
IN_CHANNELS = 1
EMBEDDING_DIM = 128

# Visualization configuration
MAX_USERS_TO_PLOT = 10  # Maximum number of users to display

print(f"Configuration:")
print(f"  Mode: {MODE}")
print(f"  Dataset: {DATASET_NAME}")
print(f"  Data path: {DATA_BASE_PATH}")
print(f"  Max samples: {MAX_SAMPLES}")
print(f"  Max users to plot: {MAX_USERS_TO_PLOT}")

Configuration:
  Mode: random
  Dataset: BiHeartS
  Data path: ../data.nosync/60s
  Max samples: 20
  Max users to plot: 10


## Create or Load Model

In [5]:
if MODE == 'checkpoint':
    # Load from checkpoint
    checkpoint_path = Path(CHECKPOINT_PATH)
    
    if not checkpoint_path.exists():
        print(f"❌ Checkpoint not found: {checkpoint_path}")
        print("Falling back to random initialization...")
        MODE = 'random'
    else:
        print(f"Loading model from checkpoint: {checkpoint_path}")
        engine = ContrastiveEngine.load_from_checkpoint(
            checkpoint_path,
            max_users_to_plot=MAX_USERS_TO_PLOT
        )
        model = engine.model
        print(f"✓ Model loaded from checkpoint")

if MODE == 'random':
    # Create random EfficientNet contrastive model
    print("Creating randomly initialized EfficientNet contrastive model...")
    
    model = EfficientNetContrastive(
        in_channels=IN_CHANNELS,
        embedding_dim=EMBEDDING_DIM,
        loss_fn=NCELoss(temperature=0.1),
    )
    
    # Create engine with the model
    engine = ContrastiveEngine(
        model=model,
        optimizer=torch.optim.Adam,
        optimizer_kwargs={'lr': 0.001},
        max_users_to_plot=MAX_USERS_TO_PLOT,
        seed=42
    )
    
    print(f"✓ EfficientNet model created with random weights")

# Set model to eval mode
model.eval()
print(f"\nModel architecture:")
print(model)

Creating randomly initialized EfficientNet contrastive model...
✓ EfficientNet model created with random weights

Model architecture:
EfficientNetContrastive(
  (stem_conv): Conv1d(1, 32, kernel_size=(3,), stride=(2,), padding=(1,), bias=False)
  (stem_bn): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (stem_activation): Swish()
  (mbconv_blocks): ModuleList(
    (0): MBConv1D(
      (expand_conv): Conv1d(32, 128, kernel_size=(1,), stride=(1,), bias=False)
      (expand_bn): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (expand_activation): Swish()
      (depthwise_conv): Conv1d(128, 128, kernel_size=(3,), stride=(2,), padding=(1,), groups=128, bias=False)
      (depthwise_bn): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (depthwise_activation): Swish()
      (se_pool): AdaptiveAvgPool1d(output_size=1)
      (se_conv1): Conv1d(128, 8, kernel_size=(1,), stride=(1,))
   

## Load Real Data from Dataset

In [6]:
# Load dataset using the built-in EDAtaNoLabel class
print(f"Loading dataset from {DATA_BASE_PATH}/{DATASET_NAME}...")

dataset = EDAtaNoLabel(
    base_path=DATA_BASE_PATH,
    dataset_names=[DATASET_NAME],
    shuffle=True,
    seed=42,
    channels=0,  # Select first channel only
)

print(f"\nDataset loaded:")
print(f"  Total samples: {len(dataset)}")
print(f"  Data shape per sample: {dataset[0]['x'].shape}")

# Extract all data and user labels
print("\nExtracting data and user labels...")
all_data = []
all_users = []

for idx in range(len(dataset)):
    sample = dataset[idx]
    all_data.append(sample['x'])
    all_users.append(sample['user'])

# Convert to tensors/arrays
data = torch.stack(all_data)
users = np.array(all_users)

# Add channel dimension if needed (for single channel data)
if data.dim() == 2:
    data = data.unsqueeze(-1)  # (N, T) -> (N, T, 1)

print(f"\nProcessed data:")
print(f"  Data shape: {data.shape}")
print(f"  Users shape: {users.shape}")
print(f"  Unique users: {len(np.unique(users))}")
print(f"  Data range: [{data.min():.3f}, {data.max():.3f}]")

Loading dataset from ../data.nosync/60s/BiHeartS...

Dataset loaded:
  Total samples: 105461
  Data shape per sample: torch.Size([240])

Extracting data and user labels...

Processed data:
  Data shape: torch.Size([105461, 240, 1])
  Users shape: (105461,)
  Unique users: 10
  Data range: [0.000, 58.508]


In [7]:
def get_random_subsample(data: np.ndarray, max_sample_size: int) -> np.ndarray:
    data_len = data.shape[0]
    random_idxs = np.random.choice(
        data_len, size=min(max_sample_size, data_len), replace=False
    )
    return data[random_idxs]


data_subsampled = [
        (get_random_subsample(data=data[users == user], max_sample_size=MAX_SAMPLES), np.array([user] * min(MAX_SAMPLES, np.sum(users == user))))
        for user in np.unique(users)
    ]
users_subsampled = np.concatenate([user for _, user in data_subsampled], axis=0)
data_subsampled = np.concatenate([x for x, _ in data_subsampled], axis=0)
data_subsampled = torch.tensor(data_subsampled)
users_subsampled = torch.tensor(users_subsampled)

## Visualize Sample Signals

In [8]:
# Plot a few example signals from different users
fig, axes = plt.subplots(3, 3, figsize=(15, 10))
axes = axes.flatten()

unique_users = np.unique(users)
num_users_to_plot = min(9, len(unique_users))
users_to_plot = np.random.choice(unique_users, size=num_users_to_plot, replace=False)

for idx, user_id in enumerate(users_to_plot):
    # Get first sample for this user
    user_mask = users == user_id
    user_indices = np.where(user_mask)[0]
    
    if len(user_indices) > 0:
        # Get the data sample - handle both 2D and 3D cases
        sample_data = data_subsampled[user_indices[0]]
        if sample_data.dim() == 2:
            # If 2D (T, C), take first channel
            sample = sample_data[:, 0].numpy()
        else:
            # If 1D (T,), use directly
            sample = sample_data.numpy()
        
        axes[idx].plot(sample)
        axes[idx].set_title(f'User {user_id}')
        axes[idx].set_xlabel('Time')
        axes[idx].set_ylabel('EDA Signal')
        axes[idx].grid(True, alpha=0.3)

# Hide unused subplots
for idx in range(num_users_to_plot, 9):
    axes[idx].axis('off')

plt.tight_layout()
plt.suptitle(f'Sample EDA Signals from {DATASET_NAME}', fontsize=16, y=1.00)
plt.show()

print(f"✓ Displayed sample signals from {num_users_to_plot} users")

✓ Displayed sample signals from 9 users


## Extract Embeddings

Use the model's `get_representation()` method to extract embeddings for all samples.

In [9]:
# Extract embeddings
print("Extracting embeddings...")

with torch.no_grad():
    embeddings = model.get_representation(data_subsampled)

print(f"\nEmbeddings:")
print(f"  Shape: {embeddings.shape}")
print(f"  Mean: {embeddings.mean():.4f}")
print(f"  Std: {embeddings.std():.4f}")
print(f"  Range: [{embeddings.min():.4f}, {embeddings.max():.4f}]")

print(f"\n✓ Embeddings extracted successfully")

Extracting embeddings...

Embeddings:
  Shape: torch.Size([200, 128])
  Mean: 0.0040
  Std: 0.0501
  Range: [-0.1474, 0.1794]

✓ Embeddings extracted successfully


## Test Visualization Method

Simulate the engine's internal visualization by calling the plotting method directly.

In [10]:
# Prepare data in the format expected by the engine
# The engine collects embeddings in a list during the epoch
embeddings_list = [embeddings]
users_list = [users_subsampled.numpy()]

# Call the plotting method
print("Creating UMAP visualization...")
print()

engine._plot_embedding_visualization(
    embeddings_list=embeddings_list,
    users_list=users_list,
    split="test"
)

Creating UMAP visualization...

Computing UMAP for 200 test embeddings...


/Users/leonardoalchieri/mambaforge/envs/eda-foundation-models/lib/python3.13/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
OMP: Info #276: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.


Saved plot to ./test_embedding_umap_step0.png
✓ UMAP visualization created for test epoch 0


## Test with Different Numbers of Users to Plot

Test the visualization with various `max_users_to_plot` configurations.

In [9]:
# Test with different max_users_to_plot values
num_unique_users = len(np.unique(users))
test_configs = [5, 10, 20, min(50, num_unique_users)]

for max_users in test_configs:
    if max_users > num_unique_users:
        continue
        
    print(f"\n{'='*80}")
    print(f"Testing with max_users_to_plot={max_users}")
    print(f"{'='*80}")
    
    # Update engine configuration
    engine.max_users_to_plot = max_users
    engine.current_epoch = max_users  # Use as identifier in title
    
    # Create visualization
    engine._plot_embedding_visualization(
        embeddings_list=embeddings_list,
        users_list=users_list,
        split="test"
    )
    
    print(f"✓ Visualization created with max_users_to_plot={max_users}")


Testing with max_users_to_plot=5


AttributeError: property 'current_epoch' of 'ContrastiveEngine' object has no setter

## Manual UMAP Visualization (for comparison)

Create a manual UMAP plot to compare with the engine's visualization.

In [ ]:
from umap import UMAP

# Flatten embeddings if needed
embeddings_np = embeddings.numpy()
if len(embeddings_np.shape) > 2:
    embeddings_np = embeddings_np.reshape(embeddings_np.shape[0], -1)

# Apply UMAP
print("Computing UMAP for manual visualization...")
umap = UMAP(n_components=2, random_state=42, n_neighbors=15)
embeddings_2d = umap.fit_transform(embeddings_np)

# Create plot with ALL users (no sampling)
fig, axes = plt.subplots(1, 2, figsize=(20, 8))

# Left plot: All users
unique_users = np.unique(users)
colors_all = plt.cm.tab20(np.linspace(0, 1, len(unique_users)))

for idx, user in enumerate(unique_users):
    mask = users == user
    axes[0].scatter(
        embeddings_2d[mask, 0],
        embeddings_2d[mask, 1],
        c=[colors_all[idx]],
        label=f'User {user}',
        alpha=0.6,
        s=50
    )

axes[0].set_xlabel('UMAP Dimension 1', fontsize=12)
axes[0].set_ylabel('UMAP Dimension 2', fontsize=12)
axes[0].set_title(f'All Users ({len(unique_users)} total)', fontsize=14, fontweight='bold')
axes[0].grid(True, alpha=0.3)

# Right plot: Sampled users (same as engine)
max_users = MAX_USERS_TO_PLOT
if len(unique_users) > max_users:
    rng = np.random.RandomState(42)
    selected_users = rng.choice(unique_users, size=max_users, replace=False)
else:
    selected_users = unique_users

mask_selected = np.isin(users, selected_users)
embeddings_2d_filtered = embeddings_2d[mask_selected]
users_filtered = users[mask_selected]

colors_sampled = plt.cm.tab20(np.linspace(0, 1, len(selected_users)))

for idx, user in enumerate(selected_users):
    mask = users_filtered == user
    axes[1].scatter(
        embeddings_2d_filtered[mask, 0],
        embeddings_2d_filtered[mask, 1],
        c=[colors_sampled[idx]],
        label=f'User {user}',
        alpha=0.6,
        s=50
    )

axes[1].set_xlabel('UMAP Dimension 1', fontsize=12)
axes[1].set_ylabel('UMAP Dimension 2', fontsize=12)
axes[1].set_title(f'Sampled Users ({len(selected_users)} shown)', fontsize=14, fontweight='bold')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n✓ Manual comparison plot created")
print(f"  Left: All {len(unique_users)} users")
print(f"  Right: {len(selected_users)} sampled users (matching engine behavior)")

## Summary

This notebook demonstrated:

1. ✅ Loading **EfficientNet contrastive model** from checkpoint or creating random ones
2. ✅ Loading **real EDA data** from dataset using `EDAtaNoLabel` class
3. ✅ Extracting embeddings using the model's `get_representation()` method
4. ✅ Testing the engine's visualization method with real user data
5. ✅ Testing with different `max_users_to_plot` values
6. ✅ Comparing with manual UMAP visualization

The visualization correctly:
- Uses the EfficientNet contrastive architecture
- Loads real data with user labels from the built-in dataset class
- Samples users when there are more than `max_users_to_plot`
- Uses consistent random seed for reproducibility
- Creates clean, interpretable plots
- Filters embeddings to only show selected users